# Breit-Wigner dynamics

<!-- cspell:ignore nstar -->

{class}`.BreitWignerBuilder` turns a {class}`.ThreeBodyDecayChain` into a propagator with optional production and decay form factors. Its options select a convention, not a different physics model, so that the downstream models in [polarimetry](https://github.com/ComPWA/polarimetry), jpsi-nstar, and [TR-036](https://github.com/ComPWA/report/pull/44) can share one implementation.

The formulas on this page are rendered from the builder itself, so they cannot drift from the code. The propagator is unfolded to show its denominator, while the form factor $\mathcal{F}_L$ and the running width $\Gamma$ stay folded; both are defined under [](#building-blocks).

In [ ]:
from itertools import product

import qrules
import sympy as sp
from ampform.dynamics import BreitWigner, SimpleBreitWigner
from ampform.dynamics.form_factor import FormFactor
from IPython.display import Latex, Markdown, Math, display

from ampform_dpd.adapter.qrules import normalize_state_ids, to_three_body_decay
from ampform_dpd.decay import State
from ampform_dpd.dynamics import BreitWignerBuilder
from ampform_dpd.io import aslatex, unfold_definitions

REACTION = qrules.generate_transitions(
    initial_state="D+",
    final_state=["pi+", "pi+", "pi-"],
    allowed_intermediate_particles=["rho(770)0", "f(0)(980)"],
    formalism="canonical-helicity",
    mass_conservation_factor=0,
)
DECAY = to_three_body_decay(normalize_state_ids(REACTION).transitions, min_ls=True)
S_WAVE_CHAIN, P_WAVE_CHAIN = DECAY.chains


def unfold_propagator(expression: sp.Expr) -> sp.Expr:
    for cls in (BreitWigner, SimpleBreitWigner):
        expression = expression.xreplace({
            bw: bw.evaluate() for bw in expression.atoms(cls)
        })
    return expression


def render(builder, chain=P_WAVE_CHAIN, header=None):
    if header is not None:
        display(Markdown(header))
    result = builder(chain)
    display(Math(sp.latex(unfold_propagator(result.expression))))
    return result

## Example decay

The examples below use $D^+ \to \pi^+\pi^+\pi^-$ with a $\rho(770)^0$ and an $f_0(980)$ resonance. The $\rho$ chain has a $P$ wave on both vertices, so it carries both form factors; the $f_0$ chain is an $S$ wave on both vertices and shows what the builder does when there is nothing to dampen.

In [ ]:
Latex(aslatex(DECAY))

(building-blocks)=
## Building blocks

The propagator, the form factor, and the energy-dependent width come from [AmpForm](https://ampform.readthedocs.io), so that the builder only has to decide how to combine them. The decay vertex is evaluated in the invariant mass $\sigma_1$ of the resonance, while the production vertex uses the running resonance mass $\sqrt{\sigma_1}$ against the parent mass $m_0$ and the spectator mass $m_1$.

In [ ]:
expression = BreitWignerBuilder()(P_WAVE_CHAIN).expression
definitions = {}
for node in sorted(expression.atoms(BreitWigner, FormFactor), key=str):
    definitions.update(unfold_definitions(node))
Latex(aslatex(definitions))

## Normalization conventions

Write $F_p$ and $F_d$ for the production and decay form factors, $F_p^0$ and $F_d^0$ for their values at the resonance pole, and $D$ for the propagator denominator. Two independent options then select one of four conventions:

| `normalize_form_factors` | `numerator` | Chain dynamics |
| --- | --- | --- |
| `False` | `"unity"` | $F_p F_d / D$ |
| `False` | `"mass_width"` | $m_R \Gamma_R F_p F_d / D$ |
| `True` | `"unity"` | $F_p F_d / (F_p^0 F_d^0 D)$ |
| `True` | `"mass_width"` | $m_R \Gamma_R F_p F_d / (F_p^0 F_d^0 D)$ |

Normalization applies to the external vertex factors only. It does not normalize the chain to unity at the pole, and it leaves the pole normalization $\Gamma(m_R^2) = \Gamma_R$ of the running width untouched. Normalizing the overall amplitude remains a matter of choosing the couplings.

In [ ]:
for normalize, numerator in product([False, True], ["unity", "mass_width"]):
    render(
        BreitWignerBuilder(normalize_form_factors=normalize, numerator=numerator),
        header=f"**`normalize_form_factors={normalize}`, `numerator={numerator!r}`**",
    )

## Blatt–Weisskopf convention

`normalize_form_factors` divides each vertex factor by its value at the pole. `blatt_weisskopf_convention` is independent of that choice: it forwards to the `normalize` option of AmpForm's {class}`~ampform.dynamics.form_factor.FormFactor`. The normalized form equals one at $z=1$, whereas the unnormalized form omits the factor $\lvert h^{(1)}_L(1) \rvert$, which is $1$, $\sqrt{2}$ and $\sqrt{13}$ for $L=0,1,2$.

The factor is constant per vertex, so it cancels against the pole value. The two conventions therefore differ only when `normalize_form_factors=False`. The $\Lambda_b^0 \to p K^- \gamma$ model in {doc}`/lb2pkg` uses the unnormalized convention.

In [ ]:
baseline = BreitWignerBuilder()(P_WAVE_CHAIN).expression
for convention in ["normalized", "unnormalized"]:
    result = render(
        BreitWignerBuilder(blatt_weisskopf_convention=convention),
        header=f"**`blatt_weisskopf_convention={convention!r}`**",
    )
assert P_WAVE_CHAIN.decay_node.interaction is not None
assert P_WAVE_CHAIN.production_node.interaction is not None
normalizations = {0: 1, 1: sp.sqrt(2), 2: sp.sqrt(13)}
expected = 1 / (
    normalizations[P_WAVE_CHAIN.decay_node.interaction.L]
    * normalizations[P_WAVE_CHAIN.production_node.interaction.L]
)
assert sp.simplify((result.expression / baseline).doit()).equals(expected)

## Width and vertex form factors

A constant width replaces $\Gamma(\sigma_1)$ by the pole width $\Gamma_R$. Switching off a vertex removes its factor completely, including the pole denominator that `normalize_form_factors` would otherwise introduce for it. The decay radius survives in the running width even when the decay form factor is switched off, because the width has a form factor of its own.

In [ ]:
for running, production, decay in product([False, True], repeat=3):
    render(
        BreitWignerBuilder(
            energy_dependent_width=running,
            production_form_factor=production,
            decay_form_factor=decay,
            normalize_form_factors=True,
        ),
        header=(
            f"**`energy_dependent_width={running}`,"
            f" `production_form_factor={production}`,"
            f" `decay_form_factor={decay}`**"
        ),
    )

## S waves

An $S$-wave vertex has a form factor of one, so the builder omits it and does not introduce a radius for it. The $f_0(980)$ chain is an $S$ wave on both vertices, so it reduces to a propagator with no external form factors and no radius parameter, whatever the normalization options say. Its width still runs, because an $S$-wave width has a phase-space factor of its own.

In [ ]:
s_wave = render(BreitWignerBuilder(normalize_form_factors=True), S_WAVE_CHAIN)
assert not s_wave.expression.atoms(FormFactor)
assert not any(str(symbol).startswith("R_") for symbol in s_wave.parameters)

## Parameters

Each chain carries suggested values for its own symbols. The external masses are fixed parameter defaults, because they are constant model inputs; only the Mandelstam variables $\sigma_i$ are event-dependent. A resonance gets its own decay radius, $R_{\rho(770)^0}$ here, shared over all its chains and $LS$ couplings, and the parent gets the production radius $R_{D^+}$. Equal default values do not identify parameters: two radii that both default to $1$ still fit independently.

In [ ]:
Latex(aslatex(BreitWignerBuilder()(P_WAVE_CHAIN)))

## Meson radii

The builder obtains the radius of each vertex with nonzero orbital angular momentum from its `meson_radius` hook, which receives the {class}`.IsobarNode` of that vertex. The default hook, {func}`.create_meson_radius_symbol`, names the radius after the parent of the vertex, so each resonance gets its own decay radius and the parent gets the production radius. A custom hook can share one radius between resonances or fix it to a number, while `parameter_defaults` sets the suggested values. An $S$-wave vertex has no form factor, so the hook is not called for it.

The example below is polarimetry's convention: normalized form factors, a unity numerator, one shared decay radius $R_\mathrm{res} = 1.5$, and a production radius $R_{\Lambda_c} = 5$.

In [ ]:
R_dec, R_prod = sp.symbols(R"R_\mathrm{res} R_{\Lambda_c}", nonnegative=True)
builder = BreitWignerBuilder(
    normalize_form_factors=True,
    meson_radius=lambda node: R_prod if isinstance(node.parent, State) else R_dec,
    parameter_defaults={R_dec: 1.5, R_prod: 5},
)
Latex(aslatex(builder(P_WAVE_CHAIN)))

## Migration

The removed `BreitWignerMinL` class corresponds to `BreitWignerBuilder(normalize_form_factors=True, numerator="unity")` with the `meson_radius` hook shown above. The other downstream conventions are:

| Consumer | `normalize_form_factors` | `numerator` |
| --- | --- | --- |
| jpsi-nstar | `False` | `"mass_width"` |
| polarimetry | `True` | `"unity"` |
| TR-036 | `False` | `"mass_width"` |

Reproducing an existing model also means reproducing its symbol names and defaults through `meson_radius` and `parameter_defaults`. Code that inspects the resulting expression has to recognize {class}`~ampform.dynamics.BreitWigner` and {class}`~ampform.dynamics.SimpleBreitWigner` instead of `BreitWignerMinL`.